In [9]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.sparse import load_npz

### 1. Row count và Split ratio

In [10]:
train = pq.read_table("../data/gold/full_load/train.parquet").to_pandas()
val   = pq.read_table("../data/gold/full_load/val.parquet").to_pandas()
test  = pq.read_table("../data/gold/full_load/test.parquet").to_pandas()

total = len(train) + len(val) + len(test)
print(f"Train: {len(train):,} ({len(train)/total:.1%})")
print(f"Val:   {len(val):,}   ({len(val)/total:.1%})")
print(f"Test:  {len(test):,}  ({len(test)/total:.1%})")
# Kỳ vọng: ~85% train+val, ~15% test (tháng holdout)

Train: 82,556 (77.3%)
Val:   14,569   (13.6%)
Test:  9,711  (9.1%)


### 2. Label balance trong từng split

In [11]:
print(f"Train spam: {train['label'].mean():.1%}")
print(f"Val   spam: {val['label'].mean():.1%}")
print(f"Test  spam: {test['label'].mean():.1%}")
# Kỳ vọng: cả 3 đều gần nhau ~50/50
# Nếu test lệch nhiều → tháng holdout có distribution khác → cần chú ý khi evaluate

Train spam: 47.0%
Val   spam: 47.0%
Test  spam: 47.2%


### 3. Sparse matrix shape và không có NaN

In [12]:
X_train = load_npz("../data/gold/full_load/train_X.npz")
X_val   = load_npz("../data/gold/full_load/val_X.npz")
X_test  = load_npz("../data/gold/full_load/test_X.npz")

print(f"X_train: {X_train.shape}")  # (n_train, 30004) — 30000 tfidf + 4 numeric
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")

# Check không có NaN trong numeric features
print(train[["log_chars","avg_word_length","unique_word_ratio","exclaim_count"]].isnull().sum())

X_train: (82556, 30004)
X_val:   (14569, 30004)
X_test:  (9711, 30004)
log_chars            0
avg_word_length      0
unique_word_ratio    0
exclaim_count        0
dtype: int64


### 4. Test set đúng là tháng holdout

In [13]:
# Xác nhận test set được chia đúng tháng chưa
print("Test months:", test["month_partition"].unique())
# Phải là đúng HOLDOUT_MONTH đã config trong gold_build.py
print("Train months:", sorted(train["month_partition"].unique()))

Test months: ['2026-03']
Train months: ['2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02']
